# Anotação e ruído de rótulos

Vamos analisar as anotações do `pvd_images`: medir a concordância entre os anotadores, procurar
viés e usar um classificador para montar uma fila de imagens com rótulo suspeito.

In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

def load_rgb(path):
    """Abre qualquer imagem do conjunto e devolve um RGB de 8 bits. Essa função foi implementada
    em um notebook anterior."""
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode in ("RGBA", "LA") or "transparency" in image.info:
            image = image.convert("RGBA")
            background = Image.new("RGBA", image.size, (255, 255, 255, 255))
            image = Image.alpha_composite(background, image)
        return image.convert("RGB")

def plot_matrix(matrix, fmt=".2f", cmap="Blues"):
    """Mostra um DataFrame como mapa de calor, com o valor escrito em cada célula."""
    fig, ax = plt.subplots(figsize=(0.7*matrix.shape[1] + 2, 0.5*matrix.shape[0] + 1))
    ax.imshow(matrix.to_numpy(), cmap=cmap, aspect="auto")
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            ax.text(column, row, format(value, fmt), ha="center", va="center", fontsize=8,
                    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.6, "pad": 1})
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_yticklabels(matrix.index)
    fig.tight_layout()
    plt.show()

def normalize_rows(features):
    """Divide cada linha pelo seu comprimento, para que todos os vetores tenham norma 1."""
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    return features / norms